# Toy example — three-state CTMC (STRUCS-26-00168, Section 3)

Clean, single-source reproduction of Figures 3–7 of the paper
*"A Generalized Framework for Performance-Based Earthquake Engineering:
Integrated Assessment of Structural Reliability and Resilience"*
(Nardin, Marelli, Sudret, Broccardo).

All numerical quantities are computed through the modules `core` (CTMC model),
`metrics` (reliability and resilience) and `spectral` (spectral analysis) of `ctmc_pbee`,
whose docstrings map every function to its equation number in the paper (Eq. 8–29). This is the same backend used for the REC2026 revised figures,
so both the plots *and* the printed numbers should match the paper.

Cells are grouped one figure at a time — run them top to bottom.

## 0. Imports and general configuration

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from scipy.interpolate import griddata
from scipy.optimize import brentq

from ctmc_pbee import core, metrics, spectral   # CTMC model, metrics (Eq. 10-19), spectral (Eq. 20-27)
from ctmc_pbee import utils             # utils.saveplot()
from ctmc_pbee import plotting          # plotting.add_discrete_colorbar()
from ctmc_pbee.paths import output_dir


# general-purpose config dict, passed to utils.saveplot(fname, general) throughout.
# Figures are written to output/toy_example/ (same names as the benchmarks in
# figures/toy_example/, without the B- prefix).
general = {}
general['saveFig']       = True
general['folderFig']     = output_dir('toy_example')

## 1. Plot style and colors

Same rcParams / palette used throughout the paper's toy-example figures.

In [ ]:
mpl.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 16,
    "figure.titlesize": 22,
    "mathtext.fontset": "stix",
})

darkred   = "#9E0000"
darkgray  = "#535353"
lightgray = "#E5E5E5"
lightred  = "#AD4242"
darkblue  = "#08407F"

darkred_darkgray = LinearSegmentedColormap.from_list("darkred_darkgray", [darkred, darkgray])

r, g, b, _ = to_rgba(darkred)
darkred_alpha = LinearSegmentedColormap.from_list("darkred_alpha", [(r, g, b, 0.2), (r, g, b, 0.7)], N=256)

r, g, b, _ = to_rgba(darkblue)
darkblue_alpha = LinearSegmentedColormap.from_list("darkblue_alpha", [(r, g, b, 0.0), (r, g, b, 1.0)], N=256)




## 2. Fixed model parameters (Table 1)

Dimensionless parametrization with `mu10` fixed to 1 [year]^-1:
`alpha = lambda01/mu10`, `gamma = lambda02/mu10`, `epsilon = lambda12/lambda01`.
The reference collapse rate corresponds to a 475-year return period (EC8).

In [ ]:
mu10 = 1.0   # [1/yr]
# The toy example works with the dimensionless generator Q_tilde = core.build_Q_adim(alpha, gamma, epsilon)
# (Eq. 28-29), which must be evaluated at the dimensionless time t_tilde = mu10 * t.
# With mu10 = 1 [1/yr], t_tilde coincides numerically with t in years.
lam02_ref = 1.0 / 475.0        # [year^-1] reference collapse rate (475-yr return period)
gamma_ref = lam02_ref / mu10   # dimensionless

## 3. Figure 3 — time evolution of state probabilities

Two extreme values of gamma from Table 1's range, with alpha = 0.5,
epsilon = 2.0, mu10 = 1, pi(0) = (1, 0, 0).

> NOTE: the exploratory notebook that produced the current
> `plots/*-long-term-probability-trajectories*-v2.svg` used `epsilon = 1.0`,
> while the paper caption states `epsilon = 2.0`. This cell follows the
> **paper caption**; change `epsilon` below to `1.0` if you want to match
> the exact SVG files already in `plots/` instead.

In [ ]:
alpha_f3   = 0.5
epsilon_f3 = 2.0          # paper caption; the last-saved SVG used 1.0 (see note above)
T_hor_f3   = 30.0
gammas_f3  = np.array([1e-6, 1e-1])   # two extremes of Table 1's gamma range

times = np.linspace(0, T_hor_f3, 201)
pi0 = np.array([1.0, 0.0, 0.0])

fig = plt.figure(figsize=(7, 3))
for g in gammas_f3:
    Q = core.build_Q_adim(alpha_f3, g, epsilon_f3)   # Q_tilde
    pis = np.array([core.pi_t(Q, mu10 * t, pi0) for t in times])
    plt.plot(times, pis[:, 0], linestyle='-',  alpha=0.9, label=fr"p(ds0), $\gamma={g:.0e}$")
    plt.plot(times, pis[:, 1], linestyle='-.', alpha=0.9, label=fr"p(ds1), $\gamma={g:.0e}$")
    plt.plot(times, pis[:, 2], linestyle='--', alpha=0.9, label=fr"p(ds2), $\gamma={g:.0e}$")

fig.legend(loc='outside lower center', ncol=3, bbox_to_anchor=(0.5, -0.25))
plt.xlim([0, T_hor_f3]); plt.ylim([0, 1.0])
plt.xticks(np.arange(0, T_hor_f3 + 1, 10))
plt.grid(); plt.xlabel('t')
plt.title(fr"State probabilities, $\alpha={alpha_f3}$, $\varepsilon={epsilon_f3}$, $\mu_{{10}}={mu10}$")
plt.tight_layout()

utils.saveplot("TE-long-term-probability-trajectories-mu10-fixed-v2", general)

## 4. Figure 4(a) — failure probability F_T(T_hor) vs gamma

In [ ]:
epsilon_f4 = 2.0
T_hor_f4   = 50.0
gammas_Fa  = np.logspace(-6, -1, 30)
alphas_Fa  = np.array([0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.07, 0.1, 1.0])
alphas_Fa_log = np.log10(alphas_Fa)
norm_Fa = plt.Normalize(vmin=alphas_Fa_log.min(), vmax=alphas_Fa_log.max())

fig1, ax1 = plt.subplots(figsize=(8, 5))
for a in alphas_Fa:
    Fs = []
    for g in gammas_Fa:
        Q_T = core.extract_Q_T(core.build_Q_adim(a, g, epsilon_f4))
        F, _ = metrics.F_tau(Q_T, mu10 * T_hor_f4)   # t_tilde = mu10 * t
        Fs.append(F)
    color = darkred_darkgray(norm_Fa(np.log10(a)))
    ax1.semilogx(gammas_Fa, Fs, color=color, lw=1.5)

plotting.add_discrete_colorbar(fig1, ax1, darkred_darkgray, alphas_Fa_log, label=r"$\alpha$")
ax1.set_xlabel(r"$\gamma = \lambda_{02}/\mu_{10}$")
ax1.set_ylabel(r"$\mathcal{F}_\mathcal{T}(T_\mathrm{hor})$")
ax1.set_title(fr"Failure probability vs $\gamma$  ($\varepsilon={epsilon_f4}$, $T_\mathrm{{hor}}={T_hor_f4:.0f}$ yr)")
ax1.set_xlim(gammas_Fa.min(), gammas_Fa.max()); ax1.set_ylim(0, 1)
ax1.grid(True, which="both", ls="--", alpha=0.4)
plt.setp(ax1.spines.values(), linewidth=1.5)
fig1.tight_layout()

utils.saveplot("mTE-ftau-function-gamma-mu10-fixed", general)

## 5. Figure 4(b) — reliability index beta vs gamma

Same sweep, showing beta = -Phi^-1(F). The dashed vertical line marks the
EC8 reference collapse rate (475-year return period).

In [ ]:
gammas_Fb = np.logspace(-6, -1, 50)
alphas_Fb = np.logspace(-6, -1, 20)
alphas_Fb_log = np.log10(alphas_Fb)
norm_Fb = plt.Normalize(vmin=alphas_Fb_log.min(), vmax=alphas_Fb_log.max())

fig2, ax2 = plt.subplots(figsize=(8, 5))
for a in alphas_Fb:
    betas = []
    for g in gammas_Fb:
        Q_T = core.extract_Q_T(core.build_Q_adim(a, g, epsilon_f4))
        F, _ = metrics.F_tau(Q_T, mu10 * T_hor_f4)   # t_tilde = mu10 * t
        betas.append(metrics.beta_index(F))
    color = darkred_darkgray(norm_Fb(np.log10(a)))
    ax2.semilogx(gammas_Fb, betas, color=color, lw=1.2)

ax2.axvline(gamma_ref, color="black", lw=1.2, ls="--",
            label=r"$\lambda_{02}\approx1/475\,\mathrm{yr}^{-1}$")
plotting.add_discrete_colorbar(fig2, ax2, darkred_darkgray, alphas_Fb_log, label=r"$\alpha$")
ax2.set_xlabel(r"$\gamma = \lambda_{02}/\mu_{10}$")
ax2.set_ylabel(r"$\beta$")
ax2.set_title(fr"Reliability index $\beta$ vs $\gamma$  ($\varepsilon={epsilon_f4}$, $T_\mathrm{{hor}}={T_hor_f4:.0f}$ yr)")
ax2.set_xlim(gammas_Fb.min(), gammas_Fb.max()); ax2.set_ylim(-2, 4)
ax2.grid(True, which="both", ls="--", alpha=0.4)
ax2.legend(fontsize=9, loc="upper right")
plt.setp(ax2.spines.values(), linewidth=1.5)
fig2.tight_layout()

utils.saveplot("fig_beta", general)

## 6. Figure 5 — beta(gamma, alpha) contour maps for two time horizons

In [ ]:
epsilon_f5 = 2.0
gammas_f5 = np.logspace(-6, -1, 60)
alphas_f5 = np.logspace(-6, -1, 60)
T_horizons_f5 = [1.0, 50.0]

G, A = np.meshgrid(gammas_f5, alphas_f5)
beta_levels = np.linspace(-3, 6, 25)
iso_levels  = [-1, 0, 1, 2, 3, 4, 5]

beta_maps = {}
for T_hor in T_horizons_f5:
    B = np.zeros_like(G)
    for i, a in enumerate(alphas_f5):
        for j, g in enumerate(gammas_f5):
            Q_T = core.extract_Q_T(core.build_Q_adim(a, g, epsilon_f5))
            F, _ = metrics.F_tau(Q_T, mu10 * T_hor)   # t_tilde = mu10 * t
            B[i, j] = metrics.beta_index(F)
    beta_maps[T_hor] = np.nan_to_num(B, nan=np.nanmax(B))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for idx, T_hor in enumerate(T_horizons_f5):
    ax = axes[idx]
    B = beta_maps[T_hor]
    B_masked = np.ma.masked_invalid(B)

    pts = np.column_stack([G.ravel(), A.ravel()])
    valid = ~np.isnan(B.ravel())
    B_filled = griddata(pts[valid], B.ravel()[valid], (G, A), method='nearest')

    ctf = ax.contourf(G, A, B_masked, levels=beta_levels, cmap=darkred_alpha, extend='both')
    ax.contour(G, A, B_masked, levels=iso_levels, colors='black', linewidths=1.0)
    cs_dash = ax.contour(G, A, B_filled, levels=iso_levels, colors='black',
                          linewidths=0.7, linestyles='dashed')
    ax.clabel(cs_dash, inline=True, fontsize=14, fmt=r"$\beta = %.0f$")

    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(fr"$\beta(\gamma,\alpha)$ — $T_\mathrm{{hor}} = {T_hor:.0f}$")
    ax.set_xlabel(r"$\gamma = \lambda_{02}/\mu_{10}$")
    ax.set_ylabel(r"$\alpha = \lambda_{01}/\mu_{10}$")

left, right = axes[0].get_position().x0, axes[-1].get_position().x1
cbar_ax = fig.add_axes([left, 0.0005, right - left, 0.025])
cbar = fig.colorbar(ctf, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r"$\beta(\gamma,\alpha) = -\Phi^{-1}(\mathcal{F}(T_\mathrm{hor}))$")
plt.tight_layout()

utils.saveplot("TE-beta-function-gamma-alpha-mu10-fixed-transparency", general)

## 7. Figure 6(a) — conditional occupation probability and R0(t)

Illustrates convergence of the conditional distribution to the
quasi-stationary distribution (QSD), with R0(t) on a twin axis.
Parameters chosen for visual clarity (conceptual figure, Section 3).

In [ ]:
alpha_f6, epsilon_f6, gamma_f6 = 0.8, 0.2, 1e-3
T_hor_f6, T_long_f6 = 2.0, 8.0
N_QUAD, N_R0, N_PI = 200, 60, 3000

Q6   = core.build_Q_adim(alpha_f6, gamma_f6, epsilon_f6)   # Q_tilde, time t_tilde = mu10 * t
Q_T6 = core.extract_Q_T(Q6)
spec = spectral.spectral_analysis(Q_T6)

q0, q1, w0 = spec["q0"], 1.0 - spec["q0"], spec["w0"]
R0_sp = spectral.R0_spectral(spec)                          # Eq. 27 asymptote
R0_ex = metrics.R0_exact(Q_T6, mu10 * T_hor_f6, n_quad=N_QUAD)     # Eq. 18 exact
rho_ex = metrics.rho_index(R0_ex)                           # Eq. 19

ts_pi = np.linspace(1e-3, T_long_f6, N_PI)
pi0t, pi1t = [], []
for t in ts_pi:
    pi_T = spectral.pi_T_eigenmode(Q_T6, mu10 * t, spec=spec)
    S = pi_T.sum()
    if S > 1e-12:
        pi0t.append(pi_T[0] / S); pi1t.append(pi_T[1] / S)
    else:
        pi0t.append(q0); pi1t.append(q1)
pi0t, pi1t = np.array(pi0t), np.array(pi1t)
mask_hor, mask_long = ts_pi <= T_hor_f6, ts_pi > T_hor_f6

ts_R0 = np.linspace(0.05, T_long_f6, N_R0)
R0t_vals = np.array([metrics.R0_exact(Q_T6, mu10 * t, n_quad=N_QUAD) for t in ts_R0])

col0, col1, colT, colR0, colRsp = "#2166ac", "#92c5de", "#333333", "#4dac26", "#d01c8b"

fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax2 = ax1.twinx()

ax1.fill_between(ts_pi[mask_hor], 0, pi0t[mask_hor], color=col0, alpha=0.40,
                  label=r"$\tilde{\pi}_0(t)=P(ds_0\mid\mathcal{T}>t)$")
ax1.fill_between(ts_pi[mask_hor], pi0t[mask_hor], 1.0, color=col1, alpha=0.50,
                  label=r"$\tilde{\pi}_1(t)=P(ds_1\mid\mathcal{T}>t)$")
ax1.fill_between(ts_pi[mask_long], 0, pi0t[mask_long], color=col0, alpha=0.10)
ax1.fill_between(ts_pi[mask_long], pi0t[mask_long], 1.0, color=col1, alpha=0.12)
ax1.plot(ts_pi, pi0t, color=col0, lw=2.2)
ax1.plot(ts_pi, pi1t, color=col1, lw=2.2, ls="--")
ax1.axhline(q0, color=col0, lw=1.3, ls=":", label=fr"$q_0={q0:.2f}$")
ax1.axhline(q1, color=col1, lw=1.3, ls=":", label=fr"$q_1={q1:.2f}$")
ax1.axvline(T_hor_f6, color=colT, lw=1.6, ls="--", label=r"$T_\mathrm{hor}$")

ax2.plot(ts_R0, R0t_vals, color=colR0, lw=2.4, label=r"$\mathcal{R}_0(t)$ exact (Eq. 17)")  # legend as in the published Fig. 6a; paper numbering: Eq. 18
ax2.scatter([T_hor_f6], [R0_ex], color=colR0, s=60,
            label=fr"$\mathcal{{R}}_0(T_\mathrm{{hor}})={R0_ex:.3f}$")
ax2.axhline(R0_sp, color=colRsp, lw=1.5, ls=(0, (5, 3)),
            label=fr"$\mathcal{{R}}_0^\mathrm{{sp}}=1-w_0 q_0={R0_sp:.3f}$ (Eq. 27)")

ax1.set_xlim(0, T_long_f6); ax1.set_ylim(-0.02, 1.08); ax2.set_ylim(-0.02, 1.08)
ax1.set_xlabel("Time [years]")
ax1.set_ylabel(r"Conditional probability $\tilde{\pi}_i(t)$")
ax2.set_ylabel(r"Resilience metric $\mathcal{R}_0(t)$", color=colR0)
ax2.tick_params(axis="y", colors=colR0)
ax1.set_title(fr"$\alpha={alpha_f6}$, $\gamma={gamma_f6}$, $\varepsilon={epsilon_f6}$, "
              fr"$\mu_{{10}}={mu10}$, $T_\mathrm{{hor}}={T_hor_f6:.0f}$ yr")
ax1.grid(True, ls="--", alpha=0.28)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=9.5, loc="center right")
plt.tight_layout()

utils.saveplot("fig_R0_conditional_conceptual", general)

print(f"q0={q0:.4f}  q1={q1:.4f}  w0={w0:.4f}")
print(f"R0 exact (Eq.18) @ T_hor = {R0_ex:.4f}   rho = {rho_ex:.3f}")
print(f"R0 spectral (Eq.27)      = {R0_sp:.4f}   |error| = {abs(R0_sp - R0_ex):.2e}")

## 8. Figure 6(b) — resilience index rho vs epsilon

In [ ]:
epsilons_f6b = np.logspace(-3, 2, 50)
gamma_f6b = 0.1 * lam02_ref                 # reduced collapse rate, fixed for this sweep
alphas_f6b = np.logspace(-6, -1, 20)
alphas_f6b_log = np.log10(alphas_f6b)
norm_f6b = plt.Normalize(vmin=alphas_f6b_log.min(), vmax=alphas_f6b_log.max())

fig3, ax3 = plt.subplots(figsize=(8, 5))
for a in alphas_f6b:
    rhos = []
    for eee in epsilons_f6b:
        Q_T = core.extract_Q_T(core.build_Q_adim(a, gamma_f6b, eee))
        R0 = metrics.R0_exact(Q_T, mu10 * T_hor_f4, n_quad=100)   # T_hor = 50 yr (Sec. 4)
        rhos.append(metrics.rho_index(R0))
    color = darkred_darkgray(norm_f6b(np.log10(a)))
    ax3.semilogx(epsilons_f6b, rhos, color=color, lw=1.2)

plotting.add_discrete_colorbar(fig3, ax3, darkred_darkgray, alphas_f6b_log, label=r"$\alpha$")
ax3.set_xlabel(r"$\varepsilon = \lambda_{12}/\lambda_{01}$")
ax3.set_ylabel(r"$\rho$")
ax3.set_title(fr"Resilience index $\rho$ vs $\varepsilon$  ($\gamma={gamma_f6b:.3g}$, $T_\mathrm{{hor}}={T_hor_f4:.0f}$ yr)")
ax3.set_xlim(epsilons_f6b.min(), epsilons_f6b.max()); ax3.set_ylim(-2, 4)
ax3.grid(True, which="both", ls="--", alpha=0.4)
plt.setp(ax3.spines.values(), linewidth=1.5)
fig3.tight_layout()

utils.saveplot("fig_rho", general)

## 9. Figure 7 — joint (beta, rho) performance space

Three structural configurations with fixed absolute rates
`(lambda01, lambda02, lambda12)`, traced as `mu10` increases from
0.1 to 100 [year]^-1 (triangle -> square). The three fixed-rate sets are
**derived from the beta = 1, 2, 3 reference points of Figure 5**
(fixed `alpha_ref`, increasing `gamma`, at `mu10 = 1`), so the gold stars
fall exactly on the mu10 = 1 point of each curve, by construction —
reproducing the paper's cross-reference between Figures 5 and 7.

In [ ]:
alpha_ref = 0.001         # fixed alpha used to anchor the beta=1,2,3 points (as in Fig. 5)
epsilon_ref = 2.0
T_hor_f7 = 50.0

# at alpha_ref=0.001: beta -> ~3.7 as gamma -> 0, beta ~ -2.5 at gamma = 0.1 (Table 1 range)
target_betas = [1.0, 2.0, 3.0]
anchor_gammas = [brentq(lambda g, tb=tb: core.beta_from_dimensionless(alpha_ref, g, epsilon_ref, 1.0, T_hor_f7) - tb, 1e-8, 0.1) for tb in target_betas]

anchors = []
for tb, g in zip(target_betas, anchor_gammas):
    lam01 = alpha_ref * 1.0
    lam02 = g * 1.0
    lam12 = epsilon_ref * lam01
    Q_T = core.extract_Q_T(core.build_Q_adim(alpha_ref, g, epsilon_ref))   # anchors at mu10 = 1
    R0 = metrics.R0_exact(Q_T, 1.0 * T_hor_f7, n_quad=100)   # t_tilde = mu10 * t, mu10 = 1
    anchors.append(dict(beta=tb, gamma=g, lam01=lam01, lam02=lam02, lam12=lam12,
                         rho=metrics.rho_index(R0)))
    print(f"beta={tb:.0f}  ->  gamma={g:.3e}  (lam01={lam01:.3g}, lam02={lam02:.3g}, "
          f"lam12={lam12:.3g})  rho(mu10=1)={anchors[-1]['rho']:.3f}")

In [ ]:
mu10_vals = np.logspace(np.log10(0.1), np.log10(100), 100)   # 0.1 -> 100 [year]^-1
mu10_dots = np.logspace(np.log10(0.1), np.log10(100), 10)
colors = ["steelblue", "forestgreen", "firebrick"]
cmaps  = [plt.cm.Blues, plt.cm.Greens, plt.cm.Reds]

fig, ax = plt.subplots(figsize=(8, 6.5))
for anchor, col, cmap in zip(anchors, colors, cmaps):
    bv, rv = core.beta_rho_vs_mu10_from_rates(anchor["lam01"], anchor["lam02"], anchor["lam12"], mu10_vals, T_hor_f7)
    bd, rd = core.beta_rho_vs_mu10_from_rates(anchor["lam01"], anchor["lam02"], anchor["lam12"], mu10_dots, T_hor_f7)
    norm = plt.Normalize(vmin=np.log10(mu10_vals.min()), vmax=np.log10(mu10_vals.max()))

    ax.plot(bv, rv, color=col, lw=1.8, alpha=0.85)
    ax.scatter(bd, rd, c=np.log10(mu10_dots), s=70, cmap=cmap, norm=norm,
               edgecolor=col, linewidth=0.6)
    ax.scatter(bv[0], rv[0], s=180, marker="^", color=col, edgecolor="white", zorder=6)
    ax.scatter(bv[-1], rv[-1], s=180, marker="s", color=col, edgecolor="white", zorder=6)
    ax.plot([], [], color=col, lw=2.5,
            label=fr"$\beta_{{\mu_{{10}}=1}}={anchor['beta']:.0f}$, "
                  fr"$\lambda_{{01}}={anchor['lam01']:.2g},\,"
                  fr"\lambda_{{02}}={anchor['lam02']:.2g},\,"
                  fr"\lambda_{{12}}={anchor['lam12']:.2g}$")

# gold stars: beta=1,2,3 reference points from Figure 5, at mu10 = 1
ax.scatter([a["beta"] for a in anchors], [a["rho"] for a in anchors],
           marker="*", s=320, color="gold", edgecolor="black", linewidth=0.8, zorder=8,
           label=r"Fig. 5 reference points ($\mu_{10}=1$)")

ax.set_xlim(0, 4); ax.set_ylim(0, 4)
ax.set_xlabel(r"Reliability index $\beta$")
ax.set_ylabel(r"Resilience index $\rho$")
ax.set_title(fr"$(\beta,\rho)$ plane, $T_\mathrm{{hor}}={T_hor_f7:.0f}$ yr — "
             r"$\blacktriangle=\mu_{10}{=}0.1$, $\blacksquare=\mu_{10}{=}100$")
ax.grid(True, ls="--", alpha=0.35)
ax.legend(fontsize=8, loc="lower right", framealpha=0.92)
plt.tight_layout()

utils.saveplot("TE-overlapping", general)